In [1]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import *
from pyspark.sql.window import Window

In [2]:
spark = SparkSession.builder.appName("Capstone1pyspark").getOrCreate()

In [3]:
%%writefile stores.csv
store_id,store_name,city,state,store_type,manager_name
S101,Metro Mart Hyderabad,Hyderabad,Telangana,Supermarket,Rahul Sharma
S102,Metro Mart Bangalore,Bangalore,Karnataka,Supermarket,Priya Reddy
S103,Metro Mart Mumbai,Mumbai,Maharashtra,Hypermarket,Amit Kumar
S104,Metro Mart Chennai,Chennai,Tamil Nadu,Supermarket,Sneha Patel
S105,Metro Mart Delhi,Delhi,Delhi,Hypermarket,Farhan Ali
S106,Metro Mart Pune,Pune,Maharashtra,Mini Store,Neha Singh
S107,Metro Mart Kochi,Kochi,Kerala,Mini Store,Arjun Verma
S108,Metro Mart Jaipur,Jaipur,Rajasthan,Supermarket,Meera Nair

Overwriting stores.csv


In [4]:
%%writefile products.csv
product_id,product_name,category,brand,supplier_id,unit_price
P101,Laptop,Electronics,Lenovo,S201,65000
P102,Mobile,Electronics,Samsung,S202,25000
P103,Television,Electronics,LG,S203,45000
P104,Office Chair,Furniture,Featherlite,S204,7000
P105,Study Table,Furniture,Urban Ladder,S204,12000
P106,Shoes,Fashion,Nike,S205,4500
P107,Watch,Fashion,Fastrack,S206,8000
P108,Backpack,Fashion,Wildcraft,S206,2500
P109,Refrigerator,Electronics,Whirlpool,S203,38000
P110,Sofa,Furniture,Godrej,S204,32000
P111,Headphones,Electronics,Sony,S999,3000
P112,T-Shirt,Fashion,Puma,,1500

Overwriting products.csv


In [5]:
%%writefile inventory.csv
inventory_id,store_id,product_id,stock_quantity,reorder_level,last_update
I1001,S101,P101,10,5,2026-01-10
I1002,S101,P102,25,10,2026-01-10
I1003,S101,P104,3,5,2026-01-11
I1004,S102,P101,8,5,2026-01-12
I1005,S102,P103,5,4,2026-01-12
I1006,S103,P105,2,5,2026-01-13
I1007,S103,P106,30,10,2026-01-14
I1008,S104,P107,4,5,2026-01-15
I1009,S105,P108,50,20,2026-01-15
I1010,S106,P109,,6,2026-01-16
I1011,S107,P110,1,3,2026-01-17
I1012,S108,P120,12,5,2026-01-18

Overwriting inventory.csv


In [6]:
%%writefile sales.csv
sale_id,store_id,product_id,sale_date,quantity_sold,sale_amount,payment_mode
SA1001,S101,P101,2026-01-10,1,65000,UPI
SA1002,S101,P102,2026-01-10,2,50000,Card
SA1003,S102,P101,2026-01-11,1,65000,UPI
SA1004,S103,P106,2026-01-12,4,18000,Cash
SA1005,S104,P107,2026-01-12,1,8000,Card
SA1006,S105,P108,2026-01-13,5,12500,UPI
SA1007,S106,P109,2026-01-14,1,38000,Card
SA1008,S107,P110,2026-01-15,1,32000,UPI
SA1009,S108,P120,2026-01-15,2,10000,Cash
SA1010,S101,P104,2026-01-16,2,14000,
SA1011,S102,P103,2026-01-17,1,,UPI
SA1012,S103,P105,2026-01-18,1,12000,Card
SA1013,S104,P107,2026-02-01,2,16000,UPI
SA1014,S105,P108,2026-02-02,3,7500,Cash
SA1015,S101,P102,2026-02-03,1,25000,Card

Overwriting sales.csv


In [7]:
%%writefile suppliers.json
[
 {
  "supplier_id": "S201",
  "supplier_name": "TechSource India",
  "city": "Hyderabad",
  "rating": 4.5,
  "contact": {
   "phone": "9876500011",
   "email": "techsource@mail.com"
  }
 },
 {
  "supplier_id": "S202",
  "supplier_name": "MobileWorld Distributors",
  "city": "Bangalore",
  "rating": 4.2,
  "contact": {
   "phone": null,
   "email": "mobileworld@mail.com"
  }
 },
 {
  "supplier_id": "S203",
  "supplier_name": "HomeTech Supply",
  "city": "Mumbai",
  "rating": 4.4,
  "contact": {
   "phone": "9876500013",
   "email": null
  }
 },
 {
  "supplier_id": "S204",
  "supplier_name": "Urban Furniture Co",
  "city": "Delhi",
  "rating": 4.0,
  "contact": {
   "phone": "9876500014",
   "email": "urban@mail.com"
  }
 },
 {
  "supplier_id": "S205",
  "supplier_name": "Fashion Direct",
  "city": "Pune",
  "rating": 3.8,
  "contact": {
   "phone": null,
   "email": null
  }
 }
]

Overwriting suppliers.json


In [8]:
storedf = spark.read.csv("stores.csv", header=True, inferSchema=True)

In [9]:
productdf = spark.read.csv("products.csv", header=True, inferSchema=True)

In [10]:
inventdf = spark.read.csv("inventory.csv", header=True, inferSchema=True)

In [11]:
salesdf = spark.read.csv("sales.csv", header=True, inferSchema=True)

In [12]:
supplydf = spark.read.option("multiline", "true").json("suppliers.json")

In [13]:
storedf.printSchema()

root
 |-- store_id: string (nullable = true)
 |-- store_name: string (nullable = true)
 |-- city: string (nullable = true)
 |-- state: string (nullable = true)
 |-- store_type: string (nullable = true)
 |-- manager_name: string (nullable = true)



In [14]:
productdf.printSchema()

root
 |-- product_id: string (nullable = true)
 |-- product_name: string (nullable = true)
 |-- category: string (nullable = true)
 |-- brand: string (nullable = true)
 |-- supplier_id: string (nullable = true)
 |-- unit_price: integer (nullable = true)



In [15]:
inventdf.printSchema()

root
 |-- inventory_id: string (nullable = true)
 |-- store_id: string (nullable = true)
 |-- product_id: string (nullable = true)
 |-- stock_quantity: integer (nullable = true)
 |-- reorder_level: integer (nullable = true)
 |-- last_update: date (nullable = true)



In [16]:
salesdf.printSchema()

root
 |-- sale_id: string (nullable = true)
 |-- store_id: string (nullable = true)
 |-- product_id: string (nullable = true)
 |-- sale_date: date (nullable = true)
 |-- quantity_sold: integer (nullable = true)
 |-- sale_amount: integer (nullable = true)
 |-- payment_mode: string (nullable = true)



In [17]:
supplydf.printSchema()

root
 |-- city: string (nullable = true)
 |-- contact: struct (nullable = true)
 |    |-- email: string (nullable = true)
 |    |-- phone: string (nullable = true)
 |-- rating: double (nullable = true)
 |-- supplier_id: string (nullable = true)
 |-- supplier_name: string (nullable = true)



In [18]:
print("Stores:", storedf.count())

Stores: 8


In [19]:
print("Products:", productdf.count())

Products: 12


In [20]:
print("Inventory:", inventdf.count())

Inventory: 12


In [21]:
print("Sales:", salesdf.count())

Sales: 15


In [22]:
print("Suppliers:", supplydf.count())

Suppliers: 5


In [23]:
storedf.write.mode("overwrite").parquet("bronze/stores")

In [24]:
productdf.write.mode("overwrite").parquet("bronze/products")

In [25]:
inventdf.write.mode("overwrite").parquet("bronze/inventory")

In [26]:
salesdf.write.mode("overwrite").parquet("bronze/sales")

In [27]:
supplydf.write.mode("overwrite").parquet("bronze/suppliers")

In [28]:
productdf.filter(col("supplier_id").isNull()).show()

+----------+------------+--------+-----+-----------+----------+
|product_id|product_name|category|brand|supplier_id|unit_price|
+----------+------------+--------+-----+-----------+----------+
|      P112|     T-Shirt| Fashion| Puma|       NULL|      1500|
+----------+------------+--------+-----+-----------+----------+



In [29]:
inventdf.filter(col("stock_quantity").isNull()).show()

+------------+--------+----------+--------------+-------------+-----------+
|inventory_id|store_id|product_id|stock_quantity|reorder_level|last_update|
+------------+--------+----------+--------------+-------------+-----------+
|       I1010|    S106|      P109|          NULL|            6| 2026-01-16|
+------------+--------+----------+--------------+-------------+-----------+



In [30]:
salesdf.filter(col("sale_amount").isNull()).show()

+-------+--------+----------+----------+-------------+-----------+------------+
|sale_id|store_id|product_id| sale_date|quantity_sold|sale_amount|payment_mode|
+-------+--------+----------+----------+-------------+-----------+------------+
| SA1011|    S102|      P103|2026-01-17|            1|       NULL|         UPI|
+-------+--------+----------+----------+-------------+-----------+------------+



In [31]:
salesdf.filter(col("payment_mode").isNull()).show()

+-------+--------+----------+----------+-------------+-----------+------------+
|sale_id|store_id|product_id| sale_date|quantity_sold|sale_amount|payment_mode|
+-------+--------+----------+----------+-------------+-----------+------------+
| SA1010|    S101|      P104|2026-01-16|            2|      14000|        NULL|
+-------+--------+----------+----------+-------------+-----------+------------+



In [32]:
inventdf = inventdf.fillna({"stock_quantity": 0})

In [33]:
salesdf = salesdf.fillna({"sale_amount": 0})

In [34]:
salesdf = salesdf.fillna({"payment_mode": "Not Provided"})

In [35]:
productdf = productdf.fillna({"supplier_id": "UNKNOWN"})

In [36]:
productdf = productdf.withColumn("data_quality_status", when(col("supplier_id") == "UNKNOWN", "Invalid").otherwise("Valid"))

In [37]:
inventdf = inventdf.withColumn("data_quality_status", when(col("stock_quantity") == 0, "Needs Review").otherwise("Valid"))

In [38]:
salesdf = salesdf.withColumn("data_quality_status", when(col("sale_amount") == 0, "Needs Review").otherwise("Valid"))

In [39]:
productdf.write.mode("overwrite").parquet("silver/products")

In [40]:
inventdf.write.mode("overwrite").parquet("silver/inventory")

In [41]:
salesdf.write.mode("overwrite").parquet("silver/sales")

In [42]:
supplydf = supplydf.select("supplier_id", "supplier_name", "city", "rating", col("contact.phone").alias("phone"), col("contact.email").alias("email"))

In [43]:
supplydf = supplydf.fillna({"phone": "Not Provided"})

In [44]:
supplydf = supplydf.fillna({"email": "Not Provided"})

In [45]:
supplydf.write.mode("overwrite").parquet("silver/suppliers")

In [46]:
productsupplierdf = productdf.join(supplydf, "supplier_id", "left")

In [47]:
inventoryproductdf = inventdf.join(productdf, "product_id", "left")

In [48]:
inventoryproductdf.show()

+----------+------------+--------+--------------+-------------+-----------+-------------------+------------+-----------+------------+-----------+----------+-------------------+
|product_id|inventory_id|store_id|stock_quantity|reorder_level|last_update|data_quality_status|product_name|   category|       brand|supplier_id|unit_price|data_quality_status|
+----------+------------+--------+--------------+-------------+-----------+-------------------+------------+-----------+------------+-----------+----------+-------------------+
|      P101|       I1001|    S101|            10|            5| 2026-01-10|              Valid|      Laptop|Electronics|      Lenovo|       S201|     65000|              Valid|
|      P102|       I1002|    S101|            25|           10| 2026-01-10|              Valid|      Mobile|Electronics|     Samsung|       S202|     25000|              Valid|
|      P104|       I1003|    S101|             3|            5| 2026-01-11|              Valid|Office Chair|  Furni

In [49]:
salesstoredf = salesdf.join(storedf, "store_id", "left")

In [50]:
salesstoredf.show()

+--------+-------+----------+----------+-------------+-----------+------------+-------------------+--------------------+---------+-----------+-----------+------------+
|store_id|sale_id|product_id| sale_date|quantity_sold|sale_amount|payment_mode|data_quality_status|          store_name|     city|      state| store_type|manager_name|
+--------+-------+----------+----------+-------------+-----------+------------+-------------------+--------------------+---------+-----------+-----------+------------+
|    S101| SA1001|      P101|2026-01-10|            1|      65000|         UPI|              Valid|Metro Mart Hyderabad|Hyderabad|  Telangana|Supermarket|Rahul Sharma|
|    S101| SA1002|      P102|2026-01-10|            2|      50000|        Card|              Valid|Metro Mart Hyderabad|Hyderabad|  Telangana|Supermarket|Rahul Sharma|
|    S102| SA1003|      P101|2026-01-11|            1|      65000|         UPI|              Valid|Metro Mart Bangalore|Bangalore|  Karnataka|Supermarket| Priya

In [51]:
salesproductdf = salesdf.join(productdf, "product_id", "left")

In [52]:
salesproductdf.show()

+----------+-------+--------+----------+-------------+-----------+------------+-------------------+------------+-----------+------------+-----------+----------+-------------------+
|product_id|sale_id|store_id| sale_date|quantity_sold|sale_amount|payment_mode|data_quality_status|product_name|   category|       brand|supplier_id|unit_price|data_quality_status|
+----------+-------+--------+----------+-------------+-----------+------------+-------------------+------------+-----------+------------+-----------+----------+-------------------+
|      P101| SA1001|    S101|2026-01-10|            1|      65000|         UPI|              Valid|      Laptop|Electronics|      Lenovo|       S201|     65000|              Valid|
|      P102| SA1002|    S101|2026-01-10|            2|      50000|        Card|              Valid|      Mobile|Electronics|     Samsung|       S202|     25000|              Valid|
|      P101| SA1003|    S102|2026-01-11|            1|      65000|         UPI|              Va

In [53]:
retaildf = salesdf.join(storedf, "store_id", "left").join(productdf, "product_id", "left")

In [54]:
retaildf.show()

+----------+--------+-------+----------+-------------+-----------+------------+-------------------+--------------------+---------+-----------+-----------+------------+------------+-----------+------------+-----------+----------+-------------------+
|product_id|store_id|sale_id| sale_date|quantity_sold|sale_amount|payment_mode|data_quality_status|          store_name|     city|      state| store_type|manager_name|product_name|   category|       brand|supplier_id|unit_price|data_quality_status|
+----------+--------+-------+----------+-------------+-----------+------------+-------------------+--------------------+---------+-----------+-----------+------------+------------+-----------+------------+-----------+----------+-------------------+
|      P101|    S101| SA1001|2026-01-10|            1|      65000|         UPI|              Valid|Metro Mart Hyderabad|Hyderabad|  Telangana|Supermarket|Rahul Sharma|      Laptop|Electronics|      Lenovo|       S201|     65000|              Valid|
|   

In [55]:
productdf.join(supplydf, "supplier_id", "left_anti").show()

+-----------+----------+------------+-----------+---------+----------+-------------------+
|supplier_id|product_id|product_name|   category|    brand|unit_price|data_quality_status|
+-----------+----------+------------+-----------+---------+----------+-------------------+
|       S206|      P107|       Watch|    Fashion| Fastrack|      8000|              Valid|
|       S206|      P108|    Backpack|    Fashion|Wildcraft|      2500|              Valid|
|       S999|      P111|  Headphones|Electronics|     Sony|      3000|              Valid|
|    UNKNOWN|      P112|     T-Shirt|    Fashion|     Puma|      1500|            Invalid|
+-----------+----------+------------+-----------+---------+----------+-------------------+



In [56]:
inventdf.join(productdf, "product_id", "left_anti").show()

+----------+------------+--------+--------------+-------------+-----------+-------------------+
|product_id|inventory_id|store_id|stock_quantity|reorder_level|last_update|data_quality_status|
+----------+------------+--------+--------------+-------------+-----------+-------------------+
|      P120|       I1012|    S108|            12|            5| 2026-01-18|              Valid|
+----------+------------+--------+--------------+-------------+-----------+-------------------+



In [57]:
salesdf.join(productdf, "product_id", "left_anti").show()

+----------+-------+--------+----------+-------------+-----------+------------+-------------------+
|product_id|sale_id|store_id| sale_date|quantity_sold|sale_amount|payment_mode|data_quality_status|
+----------+-------+--------+----------+-------------+-----------+------------+-------------------+
|      P120| SA1009|    S108|2026-01-15|            2|      10000|        Cash|              Valid|
+----------+-------+--------+----------+-------------+-----------+------------+-------------------+



In [58]:
salesdf.join(storedf, "store_id", "left_anti").show()

+--------+-------+----------+---------+-------------+-----------+------------+-------------------+
|store_id|sale_id|product_id|sale_date|quantity_sold|sale_amount|payment_mode|data_quality_status|
+--------+-------+----------+---------+-------------+-----------+------------+-------------------+
+--------+-------+----------+---------+-------------+-----------+------------+-------------------+



In [59]:
inventoryproductdf = inventoryproductdf.withColumn("stock_status", when(col("stock_quantity") <= col("reorder_level"), "Reorder Required").otherwise("Sufficient Stock"))

In [60]:
inventoryproductdf.select("product_id", "stock_quantity", "reorder_level", "stock_status").show()

+----------+--------------+-------------+----------------+
|product_id|stock_quantity|reorder_level|    stock_status|
+----------+--------------+-------------+----------------+
|      P101|            10|            5|Sufficient Stock|
|      P102|            25|           10|Sufficient Stock|
|      P104|             3|            5|Reorder Required|
|      P101|             8|            5|Sufficient Stock|
|      P103|             5|            4|Sufficient Stock|
|      P105|             2|            5|Reorder Required|
|      P106|            30|           10|Sufficient Stock|
|      P107|             4|            5|Reorder Required|
|      P108|            50|           20|Sufficient Stock|
|      P109|             0|            6|Reorder Required|
|      P110|             1|            3|Reorder Required|
|      P120|            12|            5|Sufficient Stock|
+----------+--------------+-------------+----------------+



In [61]:
productdf = productdf.withColumn("price_category", when(col("unit_price") >= 50000, "Premium").when(col("unit_price") >= 10000, "Standard").otherwise("Budget"))

In [62]:
productdf.select("product_name", "unit_price", "price_category").show()

+------------+----------+--------------+
|product_name|unit_price|price_category|
+------------+----------+--------------+
|      Laptop|     65000|       Premium|
|      Mobile|     25000|      Standard|
|  Television|     45000|      Standard|
|Office Chair|      7000|        Budget|
| Study Table|     12000|      Standard|
|       Shoes|      4500|        Budget|
|       Watch|      8000|        Budget|
|    Backpack|      2500|        Budget|
|Refrigerator|     38000|      Standard|
|        Sofa|     32000|      Standard|
|  Headphones|      3000|        Budget|
|     T-Shirt|      1500|        Budget|
+------------+----------+--------------+



In [63]:
salesdf = salesdf.withColumn("revenue_category", when(col("sale_amount") >= 50000, "High Revenue").when(col("sale_amount") >= 15000, "Medium Revenue").otherwise("Low Revenue"))

In [64]:
salesdf.select("sale_id", "sale_amount", "revenue_category").show()

+-------+-----------+----------------+
|sale_id|sale_amount|revenue_category|
+-------+-----------+----------------+
| SA1001|      65000|    High Revenue|
| SA1002|      50000|    High Revenue|
| SA1003|      65000|    High Revenue|
| SA1004|      18000|  Medium Revenue|
| SA1005|       8000|     Low Revenue|
| SA1006|      12500|     Low Revenue|
| SA1007|      38000|  Medium Revenue|
| SA1008|      32000|  Medium Revenue|
| SA1009|      10000|     Low Revenue|
| SA1010|      14000|     Low Revenue|
| SA1011|          0|     Low Revenue|
| SA1012|      12000|     Low Revenue|
| SA1013|      16000|  Medium Revenue|
| SA1014|       7500|     Low Revenue|
| SA1015|      25000|  Medium Revenue|
+-------+-----------+----------------+



In [65]:
salesdf = salesdf.withColumn("month", month("sale_date"))

In [66]:
salesdf = salesdf.withColumn("year", year("sale_date"))

In [67]:
salesdf.select("sale_date", "month", "year").show()

+----------+-----+----+
| sale_date|month|year|
+----------+-----+----+
|2026-01-10|    1|2026|
|2026-01-10|    1|2026|
|2026-01-11|    1|2026|
|2026-01-12|    1|2026|
|2026-01-12|    1|2026|
|2026-01-13|    1|2026|
|2026-01-14|    1|2026|
|2026-01-15|    1|2026|
|2026-01-15|    1|2026|
|2026-01-16|    1|2026|
|2026-01-17|    1|2026|
|2026-01-18|    1|2026|
|2026-02-01|    2|2026|
|2026-02-02|    2|2026|
|2026-02-03|    2|2026|
+----------+-----+----+



In [68]:
inventoryproductdf = inventoryproductdf.withColumn("inventory_value", col("stock_quantity") * col("unit_price"))

In [69]:
inventoryproductdf.select("product_name", "stock_quantity", "unit_price", "inventory_value").show()

+------------+--------------+----------+---------------+
|product_name|stock_quantity|unit_price|inventory_value|
+------------+--------------+----------+---------------+
|      Laptop|            10|     65000|         650000|
|      Mobile|            25|     25000|         625000|
|Office Chair|             3|      7000|          21000|
|      Laptop|             8|     65000|         520000|
|  Television|             5|     45000|         225000|
| Study Table|             2|     12000|          24000|
|       Shoes|            30|      4500|         135000|
|       Watch|             4|      8000|          32000|
|    Backpack|            50|      2500|         125000|
|Refrigerator|             0|     38000|              0|
|        Sofa|             1|     32000|          32000|
|        NULL|            12|      NULL|           NULL|
+------------+--------------+----------+---------------+



In [70]:
supplydf = supplydf.withColumn("supplier_quality", when(col("rating") >= 4.5, "Excellent").when(col("rating") >= 4.0, "Good").otherwise("Average"))

In [71]:
supplydf.select("supplier_name", "rating", "supplier_quality").show()

+--------------------+------+----------------+
|       supplier_name|rating|supplier_quality|
+--------------------+------+----------------+
|    TechSource India|   4.5|       Excellent|
|MobileWorld Distr...|   4.2|            Good|
|     HomeTech Supply|   4.4|            Good|
|  Urban Furniture Co|   4.0|            Good|
|      Fashion Direct|   3.8|         Average|
+--------------------+------+----------------+



In [72]:
retaildf = salesdf.join(storedf, "store_id", "left").join(productdf, "product_id", "left")

In [73]:
storedf.groupBy("state").count().show()

+-----------+-----+
|      state|count|
+-----------+-----+
|  Karnataka|    1|
|     Kerala|    1|
| Tamil Nadu|    1|
|      Delhi|    1|
|  Rajasthan|    1|
|  Telangana|    1|
|Maharashtra|    2|
+-----------+-----+



In [74]:
productdf.groupBy("category").count().show()

+-----------+-----+
|   category|count|
+-----------+-----+
|    Fashion|    4|
|Electronics|    5|
|  Furniture|    3|
+-----------+-----+



In [75]:
productdf.groupBy("brand").count().show()

+------------+-----+
|       brand|count|
+------------+-----+
|        Nike|    1|
|        Sony|    1|
|Urban Ladder|    1|
|        Puma|    1|
|      Lenovo|    1|
| Featherlite|    1|
|     Samsung|    1|
|      Godrej|    1|
|          LG|    1|
|   Wildcraft|    1|
|    Fastrack|    1|
|   Whirlpool|    1|
+------------+-----+



In [76]:
inventoryproductdf.groupBy("store_id").agg(sum("inventory_value").alias("total_inventory_value")).show()

+--------+---------------------+
|store_id|total_inventory_value|
+--------+---------------------+
|    S105|               125000|
|    S102|               745000|
|    S106|                    0|
|    S104|                32000|
|    S107|                32000|
|    S101|              1296000|
|    S108|                 NULL|
|    S103|               159000|
+--------+---------------------+



In [77]:
inventoryproductdf.groupBy("category").agg(sum("inventory_value").alias("total_inventory_value")).show()

+-----------+---------------------+
|   category|total_inventory_value|
+-----------+---------------------+
|    Fashion|               292000|
|       NULL|                 NULL|
|Electronics|              2020000|
|  Furniture|                77000|
+-----------+---------------------+



In [78]:
inventoryproductdf.filter(col("stock_status") == "Reorder Required").count()

5

In [79]:
salesdf.agg(sum("sale_amount").alias("total_revenue")).show()

+-------------+
|total_revenue|
+-------------+
|       373000|
+-------------+



In [80]:
retaildf.groupBy("store_id", "store_name").agg(sum("sale_amount").alias("total_revenue")).show()

+--------+--------------------+-------------+
|store_id|          store_name|total_revenue|
+--------+--------------------+-------------+
|    S105|    Metro Mart Delhi|        20000|
|    S102|Metro Mart Bangalore|        65000|
|    S101|Metro Mart Hyderabad|       154000|
|    S104|  Metro Mart Chennai|        24000|
|    S103|   Metro Mart Mumbai|        30000|
|    S108|   Metro Mart Jaipur|        10000|
|    S107|    Metro Mart Kochi|        32000|
|    S106|     Metro Mart Pune|        38000|
+--------+--------------------+-------------+



In [81]:
retaildf.groupBy("city").agg(sum("sale_amount").alias("total_revenue")).show()

+---------+-------------+
|     city|total_revenue|
+---------+-------------+
|Bangalore|        65000|
|    Kochi|        32000|
|  Chennai|        24000|
|   Mumbai|        30000|
|     Pune|        38000|
|    Delhi|        20000|
|Hyderabad|       154000|
|   Jaipur|        10000|
+---------+-------------+



In [82]:
retaildf.groupBy("category").agg(sum("sale_amount").alias("total_revenue")).show()

+-----------+-------------+
|   category|total_revenue|
+-----------+-------------+
|    Fashion|        62000|
|       NULL|        10000|
|Electronics|       243000|
|  Furniture|        58000|
+-----------+-------------+



In [83]:
retaildf.groupBy("product_id","product_name").agg(sum("sale_amount").alias("total_revenue")).show()

+----------+------------+-------------+
|product_id|product_name|total_revenue|
+----------+------------+-------------+
|      P107|       Watch|        24000|
|      P102|      Mobile|        75000|
|      P108|    Backpack|        20000|
|      P109|Refrigerator|        38000|
|      P110|        Sofa|        32000|
|      P120|        NULL|        10000|
|      P101|      Laptop|       130000|
|      P104|Office Chair|        14000|
|      P103|  Television|            0|
|      P106|       Shoes|        18000|
|      P105| Study Table|        12000|
+----------+------------+-------------+



In [84]:
salesdf.groupBy("payment_mode").agg(sum("sale_amount").alias("total_revenue")).show()

+------------+-------------+
|payment_mode|total_revenue|
+------------+-------------+
|        Card|       133000|
|        Cash|        35500|
|Not Provided|        14000|
|         UPI|       190500|
+------------+-------------+



In [85]:
retaildf.groupBy("product_id","product_name").agg(sum("sale_amount").alias("total_revenue")).orderBy(desc("total_revenue")).show(1)

+----------+------------+-------------+
|product_id|product_name|total_revenue|
+----------+------------+-------------+
|      P101|      Laptop|       130000|
+----------+------------+-------------+
only showing top 1 row


In [86]:
retaildf.groupBy("store_id","store_name").agg(sum("sale_amount").alias("total_revenue")).orderBy(desc("total_revenue")).show(1)

+--------+--------------------+-------------+
|store_id|          store_name|total_revenue|
+--------+--------------------+-------------+
|    S101|Metro Mart Hyderabad|       154000|
+--------+--------------------+-------------+
only showing top 1 row


In [87]:
retaildf.groupBy("category").agg(sum("sale_amount").alias("total_revenue")).orderBy(desc("total_revenue")).show(1)

+-----------+-------------+
|   category|total_revenue|
+-----------+-------------+
|Electronics|       243000|
+-----------+-------------+
only showing top 1 row


In [88]:
productrevdf = retaildf.groupBy("product_id","product_name").agg(sum("sale_amount").alias("total_revenue"))

In [89]:
window_spec = Window.orderBy(desc("total_revenue"))

In [90]:
productrevdf.withColumn("rank", rank().over(window_spec)).show()

+----------+------------+-------------+----+
|product_id|product_name|total_revenue|rank|
+----------+------------+-------------+----+
|      P101|      Laptop|       130000|   1|
|      P102|      Mobile|        75000|   2|
|      P109|Refrigerator|        38000|   3|
|      P110|        Sofa|        32000|   4|
|      P107|       Watch|        24000|   5|
|      P108|    Backpack|        20000|   6|
|      P106|       Shoes|        18000|   7|
|      P104|Office Chair|        14000|   8|
|      P105| Study Table|        12000|   9|
|      P120|        NULL|        10000|  10|
|      P103|  Television|            0|  11|
+----------+------------+-------------+----+



In [91]:
storerevdf = retaildf.groupBy("store_id","store_name").agg(sum("sale_amount").alias("total_revenue"))

In [92]:
window_spec = Window.orderBy(desc("total_revenue"))

In [93]:
storerevdf.withColumn("rank", rank().over(window_spec)).show()

+--------+--------------------+-------------+----+
|store_id|          store_name|total_revenue|rank|
+--------+--------------------+-------------+----+
|    S101|Metro Mart Hyderabad|       154000|   1|
|    S102|Metro Mart Bangalore|        65000|   2|
|    S106|     Metro Mart Pune|        38000|   3|
|    S107|    Metro Mart Kochi|        32000|   4|
|    S103|   Metro Mart Mumbai|        30000|   5|
|    S104|  Metro Mart Chennai|        24000|   6|
|    S105|    Metro Mart Delhi|        20000|   7|
|    S108|   Metro Mart Jaipur|        10000|   8|
+--------+--------------------+-------------+----+



In [94]:
categoryrevdf = retaildf.groupBy("category","product_id","product_name").agg(sum("sale_amount").alias("total_revenue"))

In [95]:
window_spec = Window.partitionBy("category").orderBy(desc("total_revenue"))

In [96]:
categoryrevdf.withColumn("rank", rank().over(window_spec)).show()

+-----------+----------+------------+-------------+----+
|   category|product_id|product_name|total_revenue|rank|
+-----------+----------+------------+-------------+----+
|       NULL|      P120|        NULL|        10000|   1|
|Electronics|      P101|      Laptop|       130000|   1|
|Electronics|      P102|      Mobile|        75000|   2|
|Electronics|      P109|Refrigerator|        38000|   3|
|Electronics|      P103|  Television|            0|   4|
|    Fashion|      P107|       Watch|        24000|   1|
|    Fashion|      P108|    Backpack|        20000|   2|
|    Fashion|      P106|       Shoes|        18000|   3|
|  Furniture|      P110|        Sofa|        32000|   1|
|  Furniture|      P104|Office Chair|        14000|   2|
|  Furniture|      P105| Study Table|        12000|   3|
+-----------+----------+------------+-------------+----+



In [97]:
categoryrevdf.withColumn("rank", rank().over(window_spec)).filter(col("rank")==1).show()

+-----------+----------+------------+-------------+----+
|   category|product_id|product_name|total_revenue|rank|
+-----------+----------+------------+-------------+----+
|       NULL|      P120|        NULL|        10000|   1|
|Electronics|      P101|      Laptop|       130000|   1|
|    Fashion|      P107|       Watch|        24000|   1|
|  Furniture|      P110|        Sofa|        32000|   1|
+-----------+----------+------------+-------------+----+



In [98]:
categoryrevdf.withColumn("rank", rank().over(window_spec)).filter(col("rank")<=3).show()

+-----------+----------+------------+-------------+----+
|   category|product_id|product_name|total_revenue|rank|
+-----------+----------+------------+-------------+----+
|       NULL|      P120|        NULL|        10000|   1|
|Electronics|      P101|      Laptop|       130000|   1|
|Electronics|      P102|      Mobile|        75000|   2|
|Electronics|      P109|Refrigerator|        38000|   3|
|    Fashion|      P107|       Watch|        24000|   1|
|    Fashion|      P108|    Backpack|        20000|   2|
|    Fashion|      P106|       Shoes|        18000|   3|
|  Furniture|      P110|        Sofa|        32000|   1|
|  Furniture|      P104|Office Chair|        14000|   2|
|  Furniture|      P105| Study Table|        12000|   3|
+-----------+----------+------------+-------------+----+



In [99]:
storestatedf = retaildf.groupBy("state","store_id","store_name").agg(sum("sale_amount").alias("total_revenue"))

In [100]:
window_spec = Window.partitionBy("state").orderBy(desc("total_revenue"))

In [101]:
storestatedf.withColumn("rank", rank().over(window_spec)).filter(col("rank")==1).show()

+-----------+--------+--------------------+-------------+----+
|      state|store_id|          store_name|total_revenue|rank|
+-----------+--------+--------------------+-------------+----+
|      Delhi|    S105|    Metro Mart Delhi|        20000|   1|
|  Karnataka|    S102|Metro Mart Bangalore|        65000|   1|
|     Kerala|    S107|    Metro Mart Kochi|        32000|   1|
|Maharashtra|    S106|     Metro Mart Pune|        38000|   1|
|  Rajasthan|    S108|   Metro Mart Jaipur|        10000|   1|
| Tamil Nadu|    S104|  Metro Mart Chennai|        24000|   1|
|  Telangana|    S101|Metro Mart Hyderabad|       154000|   1|
+-----------+--------+--------------------+-------------+----+



In [102]:
dailyrevdf = salesdf.groupBy("sale_date").agg(sum("sale_amount").alias("daily_revenue"))

In [103]:
window_spec = Window.orderBy("sale_date")

In [104]:
dailyrevdf.withColumn("running_total", sum("daily_revenue").over(window_spec.rowsBetween(Window.unboundedPreceding,Window.currentRow))).show()

+----------+-------------+-------------+
| sale_date|daily_revenue|running_total|
+----------+-------------+-------------+
|2026-01-10|       115000|       115000|
|2026-01-11|        65000|       180000|
|2026-01-12|        26000|       206000|
|2026-01-13|        12500|       218500|
|2026-01-14|        38000|       256500|
|2026-01-15|        42000|       298500|
|2026-01-16|        14000|       312500|
|2026-01-17|            0|       312500|
|2026-01-18|        12000|       324500|
|2026-02-01|        16000|       340500|
|2026-02-02|         7500|       348000|
|2026-02-03|        25000|       373000|
+----------+-------------+-------------+



In [105]:
dailyrevdf.withColumn("previous_day_sales", lag("daily_revenue").over(window_spec)).show()

+----------+-------------+------------------+
| sale_date|daily_revenue|previous_day_sales|
+----------+-------------+------------------+
|2026-01-10|       115000|              NULL|
|2026-01-11|        65000|            115000|
|2026-01-12|        26000|             65000|
|2026-01-13|        12500|             26000|
|2026-01-14|        38000|             12500|
|2026-01-15|        42000|             38000|
|2026-01-16|        14000|             42000|
|2026-01-17|            0|             14000|
|2026-01-18|        12000|                 0|
|2026-02-01|        16000|             12000|
|2026-02-02|         7500|             16000|
|2026-02-03|        25000|              7500|
+----------+-------------+------------------+



In [106]:
window_spec = Window.orderBy("sale_date")

In [107]:
salesdf.withColumn("next_sale_amount", lead("sale_amount").over(window_spec)).show()

+-------+--------+----------+----------+-------------+-----------+------------+-------------------+----------------+-----+----+----------------+
|sale_id|store_id|product_id| sale_date|quantity_sold|sale_amount|payment_mode|data_quality_status|revenue_category|month|year|next_sale_amount|
+-------+--------+----------+----------+-------------+-----------+------------+-------------------+----------------+-----+----+----------------+
| SA1001|    S101|      P101|2026-01-10|            1|      65000|         UPI|              Valid|    High Revenue|    1|2026|           50000|
| SA1002|    S101|      P102|2026-01-10|            2|      50000|        Card|              Valid|    High Revenue|    1|2026|           65000|
| SA1003|    S102|      P101|2026-01-11|            1|      65000|         UPI|              Valid|    High Revenue|    1|2026|           18000|
| SA1004|    S103|      P106|2026-01-12|            4|      18000|        Cash|              Valid|  Medium Revenue|    1|2026|   

In [108]:
window_spec = Window.partitionBy("product_id").orderBy("sale_date")

In [109]:
salesdf.withColumn("previous_sale_amount", lag("sale_amount").over(window_spec)).filter(col("sale_amount")>col("previous_sale_amount")).show()

+-------+--------+----------+----------+-------------+-----------+------------+-------------------+----------------+-----+----+--------------------+
|sale_id|store_id|product_id| sale_date|quantity_sold|sale_amount|payment_mode|data_quality_status|revenue_category|month|year|previous_sale_amount|
+-------+--------+----------+----------+-------------+-----------+------------+-------------------+----------------+-----+----+--------------------+
| SA1013|    S104|      P107|2026-02-01|            2|      16000|         UPI|              Valid|  Medium Revenue|    2|2026|                8000|
+-------+--------+----------+----------+-------------+-----------+------------+-------------------+----------------+-----+----+--------------------+



In [110]:
storedf.createOrReplaceTempView("stores")

In [111]:
productdf.createOrReplaceTempView("products")

In [112]:
inventdf.createOrReplaceTempView("inventory")

In [113]:
salesdf.createOrReplaceTempView("sales")

In [114]:
supplydf.createOrReplaceTempView("suppliers")

In [115]:
spark.sql("select * from sales").show()

+-------+--------+----------+----------+-------------+-----------+------------+-------------------+----------------+-----+----+
|sale_id|store_id|product_id| sale_date|quantity_sold|sale_amount|payment_mode|data_quality_status|revenue_category|month|year|
+-------+--------+----------+----------+-------------+-----------+------------+-------------------+----------------+-----+----+
| SA1001|    S101|      P101|2026-01-10|            1|      65000|         UPI|              Valid|    High Revenue|    1|2026|
| SA1002|    S101|      P102|2026-01-10|            2|      50000|        Card|              Valid|    High Revenue|    1|2026|
| SA1003|    S102|      P101|2026-01-11|            1|      65000|         UPI|              Valid|    High Revenue|    1|2026|
| SA1004|    S103|      P106|2026-01-12|            4|      18000|        Cash|              Valid|  Medium Revenue|    1|2026|
| SA1005|    S104|      P107|2026-01-12|            1|       8000|        Card|              Valid|     

In [116]:
spark.sql("select category,count(*) as total_products from products group by category").show()

+-----------+--------------+
|   category|total_products|
+-----------+--------------+
|    Fashion|             4|
|Electronics|             5|
|  Furniture|             3|
+-----------+--------------+



In [117]:
spark.sql("""
select s.store_id,s.store_name,sum(sa.sale_amount) as total_revenue
from sales sa
join stores s
on sa.store_id=s.store_id
group by s.store_id,s.store_name
""").show()

+--------+--------------------+-------------+
|store_id|          store_name|total_revenue|
+--------+--------------------+-------------+
|    S105|    Metro Mart Delhi|        20000|
|    S102|Metro Mart Bangalore|        65000|
|    S101|Metro Mart Hyderabad|       154000|
|    S104|  Metro Mart Chennai|        24000|
|    S103|   Metro Mart Mumbai|        30000|
|    S108|   Metro Mart Jaipur|        10000|
|    S107|    Metro Mart Kochi|        32000|
|    S106|     Metro Mart Pune|        38000|
+--------+--------------------+-------------+



In [118]:
spark.sql("""
select s.city,sum(sa.sale_amount) as total_revenue
from sales sa
join stores s
on sa.store_id=s.store_id
group by s.city
""").show()

+---------+-------------+
|     city|total_revenue|
+---------+-------------+
|Bangalore|        65000|
|    Kochi|        32000|
|  Chennai|        24000|
|   Mumbai|        30000|
|     Pune|        38000|
|    Delhi|        20000|
|Hyderabad|       154000|
|   Jaipur|        10000|
+---------+-------------+



In [119]:
spark.sql("""
select *
from inventory
where stock_quantity<=reorder_level
""").show()

+------------+--------+----------+--------------+-------------+-----------+-------------------+
|inventory_id|store_id|product_id|stock_quantity|reorder_level|last_update|data_quality_status|
+------------+--------+----------+--------------+-------------+-----------+-------------------+
|       I1003|    S101|      P104|             3|            5| 2026-01-11|              Valid|
|       I1006|    S103|      P105|             2|            5| 2026-01-13|              Valid|
|       I1008|    S104|      P107|             4|            5| 2026-01-15|              Valid|
|       I1010|    S106|      P109|             0|            6| 2026-01-16|       Needs Review|
|       I1011|    S107|      P110|             1|            3| 2026-01-17|              Valid|
+------------+--------+----------+--------------+-------------+-----------+-------------------+



In [120]:
spark.sql("""
select sa.*
from sales sa
left join products p
on sa.product_id=p.product_id
where p.product_id is null
""").show()

+-------+--------+----------+----------+-------------+-----------+------------+-------------------+----------------+-----+----+
|sale_id|store_id|product_id| sale_date|quantity_sold|sale_amount|payment_mode|data_quality_status|revenue_category|month|year|
+-------+--------+----------+----------+-------------+-----------+------------+-------------------+----------------+-----+----+
| SA1009|    S108|      P120|2026-01-15|            2|      10000|        Cash|              Valid|     Low Revenue|    1|2026|
+-------+--------+----------+----------+-------------+-----------+------------+-------------------+----------------+-----+----+



In [121]:
spark.sql("""
select p.*
from products p
left join suppliers s
on p.supplier_id=s.supplier_id
where s.supplier_id is null
""").show()

+----------+------------+-----------+---------+-----------+----------+-------------------+--------------+
|product_id|product_name|   category|    brand|supplier_id|unit_price|data_quality_status|price_category|
+----------+------------+-----------+---------+-----------+----------+-------------------+--------------+
|      P107|       Watch|    Fashion| Fastrack|       S206|      8000|              Valid|        Budget|
|      P108|    Backpack|    Fashion|Wildcraft|       S206|      2500|              Valid|        Budget|
|      P111|  Headphones|Electronics|     Sony|       S999|      3000|              Valid|        Budget|
|      P112|     T-Shirt|    Fashion|     Puma|    UNKNOWN|      1500|            Invalid|        Budget|
+----------+------------+-----------+---------+-----------+----------+-------------------+--------------+



In [122]:
spark.sql("""
select p.product_id,p.product_name,sum(s.sale_amount) as total_revenue
from sales s
join products p
on s.product_id=p.product_id
group by p.product_id,p.product_name
order by total_revenue desc
limit 5
""").show()

+----------+------------+-------------+
|product_id|product_name|total_revenue|
+----------+------------+-------------+
|      P101|      Laptop|       130000|
|      P102|      Mobile|        75000|
|      P109|Refrigerator|        38000|
|      P110|        Sofa|        32000|
|      P107|       Watch|        24000|
+----------+------------+-------------+



In [123]:
spark.sql("""
select payment_mode,sum(sale_amount) as total_revenue
from sales
group by payment_mode
""").show()

+------------+-------------+
|payment_mode|total_revenue|
+------------+-------------+
|        Card|       133000|
|        Cash|        35500|
|Not Provided|        14000|
|         UPI|       190500|
+------------+-------------+



In [126]:
retaildf.write.mode("overwrite").parquet("gold/sales")

In [127]:
retaildf.write.mode("overwrite").partitionBy("year","month").parquet("gold/sales_partitioned")

In [128]:
incrementaldf = spark.createDataFrame([
("SA1016","S101","P101","2026-03-01",1,65000,"UPI"),
("SA1017","S102","P102","2026-03-02",2,50000,"Card"),
("SA1018","S103","P106","2026-03-03",3,13500,"Cash")
],["sale_id","store_id","product_id","sale_date","quantity_sold","sale_amount","payment_mode"])

In [129]:
incrementaldf.show()

+-------+--------+----------+----------+-------------+-----------+------------+
|sale_id|store_id|product_id| sale_date|quantity_sold|sale_amount|payment_mode|
+-------+--------+----------+----------+-------------+-----------+------------+
| SA1016|    S101|      P101|2026-03-01|            1|      65000|         UPI|
| SA1017|    S102|      P102|2026-03-02|            2|      50000|        Card|
| SA1018|    S103|      P106|2026-03-03|            3|      13500|        Cash|
+-------+--------+----------+----------+-------------+-----------+------------+



In [130]:
incrementaldf.write.mode("overwrite").csv("incremental_sales",header=True)

In [131]:
incrementalsalesdf = spark.read.csv("incremental_sales",header=True,inferSchema=True)

In [132]:
incrementalsalesdf.show()

+-------+--------+----------+----------+-------------+-----------+------------+
|sale_id|store_id|product_id| sale_date|quantity_sold|sale_amount|payment_mode|
+-------+--------+----------+----------+-------------+-----------+------------+
| SA1017|    S102|      P102|2026-03-02|            2|      50000|        Card|
| SA1018|    S103|      P106|2026-03-03|            3|      13500|        Cash|
| SA1016|    S101|      P101|2026-03-01|            1|      65000|         UPI|
+-------+--------+----------+----------+-------------+-----------+------------+



In [133]:
incrementalsalesdf = incrementalsalesdf.withColumn("month",month("sale_date"))

In [134]:
incrementalsalesdf = incrementalsalesdf.withColumn("year",year("sale_date"))

In [135]:
incrementalsalesdf.write.mode("append").parquet("silver/sales")

In [136]:
updatedsalesdf = spark.read.parquet("silver/sales")

In [137]:
updatedretaildf = updatedsalesdf.join(storedf,"store_id","left").join(productdf,"product_id","left")

In [138]:
updatedretaildf.groupBy("product_id","product_name").agg(sum("sale_amount").alias("total_revenue")).show()

+----------+------------+-------------+
|product_id|product_name|total_revenue|
+----------+------------+-------------+
|      P107|       Watch|        24000|
|      P102|      Mobile|       125000|
|      P108|    Backpack|        20000|
|      P109|Refrigerator|        38000|
|      P110|        Sofa|        32000|
|      P120|        NULL|        10000|
|      P101|      Laptop|       195000|
|      P104|Office Chair|        14000|
|      P103|  Television|            0|
|      P106|       Shoes|        31500|
|      P105| Study Table|        12000|
+----------+------------+-------------+



In [139]:
updatedretaildf.groupBy("store_id","store_name").agg(sum("sale_amount").alias("total_revenue")).show()

+--------+--------------------+-------------+
|store_id|          store_name|total_revenue|
+--------+--------------------+-------------+
|    S105|    Metro Mart Delhi|        20000|
|    S102|Metro Mart Bangalore|       115000|
|    S101|Metro Mart Hyderabad|       219000|
|    S104|  Metro Mart Chennai|        24000|
|    S103|   Metro Mart Mumbai|        43500|
|    S108|   Metro Mart Jaipur|        10000|
|    S107|    Metro Mart Kochi|        32000|
|    S106|     Metro Mart Pune|        38000|
+--------+--------------------+-------------+



In [140]:
updatedretaildf.write.mode("overwrite").partitionBy("year","month").parquet("gold/final_sales")

In [141]:
print("Original Sales Count:",salesdf.count())

Original Sales Count: 15


In [142]:
print("Updated Sales Count:",updatedsalesdf.count())

Updated Sales Count: 18


In [143]:
storereportdf = updatedretaildf.groupBy(
"store_id",
"store_name",
"city",
"state"
).agg(
count("sale_id").alias("total_sales"),
sum("sale_amount").alias("total_revenue")
)

In [144]:
storereportdf.show()

+--------+--------------------+---------+-----------+-----------+-------------+
|store_id|          store_name|     city|      state|total_sales|total_revenue|
+--------+--------------------+---------+-----------+-----------+-------------+
|    S106|     Metro Mart Pune|     Pune|Maharashtra|          1|        38000|
|    S107|    Metro Mart Kochi|    Kochi|     Kerala|          1|        32000|
|    S101|Metro Mart Hyderabad|Hyderabad|  Telangana|          5|       219000|
|    S103|   Metro Mart Mumbai|   Mumbai|Maharashtra|          3|        43500|
|    S105|    Metro Mart Delhi|    Delhi|      Delhi|          2|        20000|
|    S102|Metro Mart Bangalore|Bangalore|  Karnataka|          3|       115000|
|    S104|  Metro Mart Chennai|  Chennai| Tamil Nadu|          2|        24000|
|    S108|   Metro Mart Jaipur|   Jaipur|  Rajasthan|          1|        10000|
+--------+--------------------+---------+-----------+-----------+-------------+



In [145]:
productreportdf = updatedretaildf.groupBy(
"product_id",
"product_name",
"category",
"brand"
).agg(
sum("quantity_sold").alias("total_quantity_sold"),
sum("sale_amount").alias("total_revenue")
)

In [146]:
productreportdf.show()

+----------+------------+-----------+------------+-------------------+-------------+
|product_id|product_name|   category|       brand|total_quantity_sold|total_revenue|
+----------+------------+-----------+------------+-------------------+-------------+
|      P110|        Sofa|  Furniture|      Godrej|                  1|        32000|
|      P104|Office Chair|  Furniture| Featherlite|                  2|        14000|
|      P101|      Laptop|Electronics|      Lenovo|                  3|       195000|
|      P109|Refrigerator|Electronics|   Whirlpool|                  1|        38000|
|      P105| Study Table|  Furniture|Urban Ladder|                  1|        12000|
|      P107|       Watch|    Fashion|    Fastrack|                  3|        24000|
|      P102|      Mobile|Electronics|     Samsung|                  5|       125000|
|      P108|    Backpack|    Fashion|   Wildcraft|                  8|        20000|
|      P106|       Shoes|    Fashion|        Nike|               

In [147]:
inventoryreportdf = inventoryproductdf.select(
"store_id",
"product_id",
"product_name",
"stock_quantity",
"reorder_level",
"stock_status"
)

In [148]:
inventoryreportdf.show()

+--------+----------+------------+--------------+-------------+----------------+
|store_id|product_id|product_name|stock_quantity|reorder_level|    stock_status|
+--------+----------+------------+--------------+-------------+----------------+
|    S101|      P101|      Laptop|            10|            5|Sufficient Stock|
|    S101|      P102|      Mobile|            25|           10|Sufficient Stock|
|    S101|      P104|Office Chair|             3|            5|Reorder Required|
|    S102|      P101|      Laptop|             8|            5|Sufficient Stock|
|    S102|      P103|  Television|             5|            4|Sufficient Stock|
|    S103|      P105| Study Table|             2|            5|Reorder Required|
|    S103|      P106|       Shoes|            30|           10|Sufficient Stock|
|    S104|      P107|       Watch|             4|            5|Reorder Required|
|    S105|      P108|    Backpack|            50|           20|Sufficient Stock|
|    S106|      P109|Refrige

In [149]:
supplierreportdf = supplydf.select(
"supplier_id",
"supplier_name",
"city",
"rating",
"supplier_quality",
"phone",
"email"
)

In [150]:
supplierreportdf.show()

+-----------+--------------------+---------+------+----------------+------------+--------------------+
|supplier_id|       supplier_name|     city|rating|supplier_quality|       phone|               email|
+-----------+--------------------+---------+------+----------------+------------+--------------------+
|       S201|    TechSource India|Hyderabad|   4.5|       Excellent|  9876500011| techsource@mail.com|
|       S202|MobileWorld Distr...|Bangalore|   4.2|            Good|Not Provided|mobileworld@mail.com|
|       S203|     HomeTech Supply|   Mumbai|   4.4|            Good|  9876500013|        Not Provided|
|       S204|  Urban Furniture Co|    Delhi|   4.0|            Good|  9876500014|      urban@mail.com|
|       S205|      Fashion Direct|     Pune|   3.8|         Average|Not Provided|        Not Provided|
+-----------+--------------------+---------+------+----------------+------------+--------------------+



In [154]:
from pyspark.sql.functions import countDistinct

In [155]:
categoryreportdf = updatedretaildf.groupBy(
"category"
).agg(
countDistinct("product_id").alias("total_products"),
sum("quantity_sold").alias("total_quantity_sold"),
sum("sale_amount").alias("total_revenue")
)

In [156]:
categoryreportdf.show()

+-----------+--------------+-------------------+-------------+
|   category|total_products|total_quantity_sold|total_revenue|
+-----------+--------------+-------------------+-------------+
|    Fashion|             3|                 18|        75500|
|       NULL|             1|                  2|        10000|
|Electronics|             4|                 10|       358000|
|  Furniture|             3|                  4|        58000|
+-----------+--------------+-------------------+-------------+



In [157]:
paymentreportdf = updatedretaildf.groupBy(
"payment_mode"
).agg(
count("sale_id").alias("total_transactions"),
sum("sale_amount").alias("total_revenue")
)

In [158]:
paymentreportdf.show()

+------------+------------------+-------------+
|payment_mode|total_transactions|total_revenue|
+------------+------------------+-------------+
|        Card|                 6|       183000|
|        Cash|                 4|        49000|
|Not Provided|                 1|        14000|
|         UPI|                 7|       255500|
+------------+------------------+-------------+

